# Análisis Exploratorio de Datos (EDA)
## Detección de Psoriasis e Identificación de Genes Relevantes Mediante Redes Neuronales
**Tecnológico de Monterrey — EIC Bioingeniería**

---

Este notebook realiza un análisis exploratorio completo del dataset clínico-genómico para la detección de psoriasis. Se abordan:
- Estadísticas descriptivas (univariante)
- Análisis de valores faltantes
- Detección de valores atípicos
- Cardinalidad de variables categóricas
- Distribuciones y transformaciones
- Correlaciones y análisis bivariante
- Balance de la variable objetivo

## 0. Importación de Librerías y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Librerías cargadas correctamente ✓')

In [ ]:
# ============================================================
# INSTRUCCIÓN: Reemplaza la ruta con la ubicación de tu archivo
# Formatos soportados: .csv, .xlsx, .tsv
# ============================================================

# Opción CSV
# df = pd.read_csv('ruta/a/tu/dataset.csv')

# Opción Excel
# df = pd.read_excel('ruta/a/tu/dataset.xlsx')

# ----- SIMULACIÓN para demostración (eliminar cuando cargues tu dataset) -----
np.random.seed(42)
n = 500

# Variables clínicas
clinical_data = {
    'edad':         np.random.normal(42, 15, n).clip(10, 90).astype(int),
    'sexo':         np.random.choice(['M', 'F'], n),
    'imc':          np.random.normal(26.5, 5, n).clip(15, 50),
    'antec_fam':    np.random.choice(['Si', 'No', np.nan], n, p=[0.35, 0.60, 0.05]),
    'fumador':      np.random.choice(['Si', 'No'], n),
    'anos_sintomas':np.random.exponential(5, n).clip(0, 40),
    'score_pasi':   np.random.exponential(8, n).clip(0, 72),
    'creatinina':   np.random.normal(0.9, 0.3, n).clip(0.3, 3.0),
    'pcr':          np.random.exponential(3, n).clip(0, 50),
}

# Simular ~200 genes (expresión genómica)
gene_names = [f'GENE_{i:04d}' for i in range(1, 201)]
gene_data  = np.random.lognormal(mean=2, sigma=1.2, size=(n, 200))

# Introducir algunos valores faltantes en genes (~3%)
mask = np.random.random((n, 200)) < 0.03
gene_data[mask] = np.nan

# Variable objetivo (0=sano, 1=psoriasis) — leve desbalance
target = np.random.choice([0, 1], n, p=[0.40, 0.60])

df_genes    = pd.DataFrame(gene_data, columns=gene_names)
df_clinical = pd.DataFrame(clinical_data)
df = pd.concat([df_clinical, df_genes, pd.Series(target, name='diagnostico')], axis=1)

# Algunos atípicos extremos
df.loc[np.random.choice(n, 5), 'imc'] = np.random.uniform(60, 80, 5)
df.loc[np.random.choice(n, 3), 'pcr'] = np.random.uniform(80, 120, 3)

print(f'Dataset cargado: {df.shape[0]} registros × {df.shape[1]} variables')
print(f'  → Variables clínicas: {len(df_clinical.columns)}')
print(f'  → Variables genómicas: {len(gene_names)}')

---
## 1. Vista General del Dataset

In [ ]:
print('=== FORMA DEL DATASET ===')
print(f'Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}')

print('\n=== PRIMEROS 5 REGISTROS (variables clínicas) ===')
display(df.iloc[:, :10].head())

print('\n=== TIPOS DE DATOS ===')
tipo_resumen = df.dtypes.value_counts().reset_index()
tipo_resumen.columns = ['Tipo', 'Cantidad']
display(tipo_resumen)

In [ ]:
# Separar variables por tipo para facilitar el análisis
TARGET = 'diagnostico'

# Detectar automáticamente clínicas vs genómicas
gene_cols     = [c for c in df.columns if c.startswith('GENE_')]
clinical_cols = [c for c in df.columns if c not in gene_cols + [TARGET]]
cat_cols      = df[clinical_cols].select_dtypes(include='object').columns.tolist()
num_cols      = df[clinical_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f'Variables clínicas numéricas  : {num_cols}')
print(f'Variables clínicas categóricas: {cat_cols}')
print(f'Variables genómicas           : {len(gene_cols)} genes')
print(f'Variable objetivo             : {TARGET}')

---
## 2. Estadísticas Descriptivas
> **Pregunta EDA:** ¿Cuáles son las estadísticas resumidas del conjunto de datos?

In [ ]:
print('=== VARIABLES CLÍNICAS NUMÉRICAS ===')
desc = df[num_cols].describe().T
desc['skewness'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurt()
display(desc.round(3))

In [ ]:
print('=== VARIABLES GENÓMICAS — RESUMEN AGREGADO ===')
gene_stats = df[gene_cols].describe().T
gene_stats['skewness'] = df[gene_cols].skew()

print(f"Media global de expresión    : {gene_stats['mean'].mean():.4f}")
print(f"Desv. estándar global        : {gene_stats['std'].mean():.4f}")
print(f"Genes con alta asimetría (>1): {(gene_stats['skewness'].abs() > 1).sum()}")

# Distribución de medias de expresión por gen
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
gene_stats['mean'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Medias\nde Expresión Génica')
axes[0].set_xlabel('Media de expresión')

gene_stats['std'].hist(bins=40, ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Distribución de Desv. Estándar\nde Expresión Génica')
axes[1].set_xlabel('Desv. estándar')

gene_stats['skewness'].hist(bins=40, ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Distribución de Asimetría\nde Expresión Génica')
axes[2].set_xlabel('Skewness')
axes[2].axvline(0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('fig_gene_stats.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Análisis de Valores Faltantes
> **Pregunta EDA:** ¿Hay valores faltantes? ¿Se pueden identificar patrones de ausencia?

In [ ]:
# --- 3.1 Reporte de nulos ---
missing = pd.DataFrame({
    'Faltantes':  df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Faltantes'] > 0].sort_values('Porcentaje', ascending=False)

print(f'Total de columnas con valores faltantes: {len(missing)}')
if len(missing) > 0:
    print(f'\nTop 20 variables con más faltantes:')
    display(missing.head(20))
else:
    print('No hay valores faltantes en el dataset.')

In [ ]:
# --- 3.2 Heatmap de faltantes (variables clínicas) ---
clin_missing = df[clinical_cols + [TARGET]].isnull()

if clin_missing.any().any():
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(clin_missing.T, cmap='YlOrRd', cbar=True,
                yticklabels=True, xticklabels=False, ax=ax)
    ax.set_title('Mapa de Valores Faltantes — Variables Clínicas\n(amarillo = faltante)')
    ax.set_xlabel('Registros')
    plt.tight_layout()
    plt.savefig('fig_missing_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No hay faltantes en variables clínicas.')

In [ ]:
# --- 3.3 Porcentaje de faltantes por gen ---
gene_missing_pct = (df[gene_cols].isnull().sum() / len(df) * 100)

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(gene_missing_pct, bins=30, color='steelblue', edgecolor='white')
ax.axvline(gene_missing_pct.mean(), color='red', linestyle='--',
           label=f'Media = {gene_missing_pct.mean():.1f}%')
ax.set_title('Distribución del % de Faltantes por Gen')
ax.set_xlabel('% de valores faltantes')
ax.set_ylabel('Número de genes')
ax.legend()
plt.tight_layout()
plt.savefig('fig_gene_missing.png', dpi=150, bbox_inches='tight')
plt.show()

genes_alto_missing = (gene_missing_pct > 20).sum()
print(f'Genes con >20% de faltantes: {genes_alto_missing}')
print(f'Genes con >50% de faltantes: {(gene_missing_pct > 50).sum()}')

In [ ]:
# --- 3.4 Estrategia de imputación ---
print('=== ESTRATEGIA DE MANEJO DE VALORES FALTANTES ===')
print()

# Variables clínicas
for col in clinical_cols:
    pct = df[col].isnull().mean() * 100
    if pct > 0:
        if pct < 5:
            estrategia = 'Imputar con mediana/moda (MCAR probable)'
        elif pct < 20:
            estrategia = 'Imputar con KNN o regresión (MAR probable)'
        elif pct < 50:
            estrategia = 'Crear indicador binario + imputar'
        else:
            estrategia = 'Considerar eliminar la variable'
        print(f'  {col:25s} {pct:5.1f}%  →  {estrategia}')

# Genes
if gene_missing_pct.mean() > 0:
    print(f'\n  Genes (promedio {gene_missing_pct.mean():.1f}%)')
    if gene_missing_pct.mean() < 10:
        print('    → Imputar con la mediana de cada gen (eficiente para alta dimensionalidad)')
    else:
        print('    → Imputar con KNN o eliminar genes con >30% de faltantes')

---
## 4. Balance de la Variable Objetivo
> **Pregunta EDA:** ¿Hay desequilibrio en las clases de la variable objetivo?

In [ ]:
conteo = df[TARGET].value_counts()
pct    = df[TARGET].value_counts(normalize=True) * 100

balance_df = pd.DataFrame({'Clase': conteo.index,
                           'Conteo': conteo.values,
                           'Porcentaje': pct.values.round(2)})
balance_df['Etiqueta'] = balance_df['Clase'].map({0: 'Sano', 1: 'Psoriasis'})
display(balance_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barras
colors = ['#2196F3', '#E91E63']
axes[0].bar(balance_df['Etiqueta'], balance_df['Conteo'], color=colors, edgecolor='white', width=0.5)
for i, (cnt, pct_val) in enumerate(zip(balance_df['Conteo'], balance_df['Porcentaje'])):
    axes[0].text(i, cnt + 3, f'{cnt}\n({pct_val:.1f}%)', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Distribución de la Variable Objetivo')
axes[0].set_ylabel('Número de pacientes')

# Pie
axes[1].pie(balance_df['Conteo'], labels=balance_df['Etiqueta'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de Clases')

plt.tight_layout()
plt.savefig('fig_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

ratio = conteo.max() / conteo.min()
print(f'\nRatio de desbalance: {ratio:.2f}:1')
if ratio > 1.5:
    print('⚠ Se detecta desbalance. Considerar: SMOTE, class_weight, o submuestreo.')
else:
    print('✓ Las clases están razonablemente balanceadas.')

---
## 5. Análisis Univariante — Variables Clínicas
> **Pregunta EDA:** ¿Existen distribuciones sesgadas? ¿Hay valores atípicos?

In [ ]:
# --- 5.1 Distribuciones de variables numéricas ---
n_vars = len(num_cols)
ncols  = 3
nrows  = (n_vars + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    sk   = skew(data)
    axes[i].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(data.mean(),   color='red',    linestyle='--', label=f'Media={data.mean():.1f}')
    axes[i].axvline(data.median(), color='orange', linestyle=':',  label=f'Mediana={data.median():.1f}')
    axes[i].set_title(f'{col}\nSkewness={sk:.2f}')
    axes[i].legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuciones de Variables Clínicas Numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.2 Análisis de asimetría y necesidad de transformación ---
print('=== ANÁLISIS DE ASIMETRÍA ===')
print(f'{"Variable":<20} {"Skewness":>10} {"Recomendación"}')
print('-' * 60)

for col in num_cols:
    sk = skew(df[col].dropna())
    if abs(sk) < 0.5:
        rec = '✓ Distribución aproximadamente normal'
    elif abs(sk) < 1.0:
        rec = '↗ Asimetría moderada — evaluar transformación'
    else:
        rec = '⚠ Alta asimetría — aplicar log(x+1) o Box-Cox'
    print(f'{col:<20} {sk:>10.3f}  {rec}')

In [ ]:
# --- 5.3 Ejemplo de transformación log para variables sesgadas ---
vars_sesgadas = [c for c in num_cols if abs(skew(df[c].dropna())) > 1]

if vars_sesgadas:
    fig, axes = plt.subplots(len(vars_sesgadas), 2,
                             figsize=(12, 4 * len(vars_sesgadas)))
    if len(vars_sesgadas) == 1:
        axes = axes.reshape(1, -1)

    for i, col in enumerate(vars_sesgadas):
        original   = df[col].dropna()
        log_transf = np.log1p(original)

        axes[i, 0].hist(original,   bins=30, color='steelblue',    edgecolor='white')
        axes[i, 0].set_title(f'{col} — Original (skew={skew(original):.2f})')

        axes[i, 1].hist(log_transf, bins=30, color='mediumseagreen', edgecolor='white')
        axes[i, 1].set_title(f'{col} — Log(x+1) (skew={skew(log_transf):.2f})')

    fig.suptitle('Efecto de Transformación Logarítmica', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_log_transform.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No se detectaron variables con alta asimetría en las clínicas numéricas.')

In [ ]:
# --- 5.4 Variables categóricas — cardinalidad y frecuencias ---
print('=== CARDINALIDAD DE VARIABLES CATEGÓRICAS ===')

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for i, col in enumerate(cat_cols):
    conteo = df[col].value_counts(dropna=False)
    card   = df[col].nunique()
    pct_na = df[col].isnull().mean() * 100
    print(f'  {col}: {card} categorías únicas, {pct_na:.1f}% faltantes')
    print(f'  {conteo.to_dict()}\n')

    conteo_clean = df[col].value_counts()
    axes[i].bar(conteo_clean.index.astype(str), conteo_clean.values,
                color=sns.color_palette('Set2', len(conteo_clean)), edgecolor='white')
    axes[i].set_title(f'{col}\n(Cardinalidad={card})')
    axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('fig_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Detección de Valores Atípicos
> **Pregunta EDA:** ¿Hay valores atípicos en el conjunto de datos?

In [ ]:
# --- 6.1 Boxplots de variables clínicas numéricas ---
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots — Detección Visual de Atípicos', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 6.2 Cuantificación por IQR ---
print('=== DETECCIÓN DE ATÍPICOS POR MÉTODO IQR ===')
print(f'{"Variable":<20} {"Atípicos":>8} {"% del total":>12} {"Acción sugerida"}')
print('-' * 70)

outlier_report = {}
for col in num_cols:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out  = ((data < lb) | (data > ub)).sum()
    pct_out = n_out / len(df) * 100
    outlier_report[col] = {'n': n_out, 'pct': pct_out, 'lb': lb, 'ub': ub}

    if pct_out == 0:
        accion = 'Sin atípicos'
    elif pct_out < 1:
        accion = 'Winsorizar o eliminar (pocos casos)'
    elif pct_out < 5:
        accion = 'Winsorizar al percentil 95'
    else:
        accion = 'Revisar — pueden ser válidos (ej. genes sobreexpresados)'

    print(f'{col:<20} {n_out:>8} {pct_out:>11.2f}%  {accion}')

In [ ]:
# --- 6.3 Z-score para variables genómicas ---
from scipy.stats import zscore

gene_z    = df[gene_cols].apply(zscore, nan_policy='omit')
extreme_z = (gene_z.abs() > 3).sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(extreme_z, bins=30, color='tomato', edgecolor='white')
ax.set_title('Número de Atípicos Extremos (|Z|>3) por Gen')
ax.set_xlabel('Nº de registros con |Z-score| > 3')
ax.set_ylabel('Número de genes')
plt.tight_layout()
plt.savefig('fig_gene_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Genes con al menos 1 atípico extremo : {(extreme_z > 0).sum()}')
print(f'Genes con >5 atípicos extremos        : {(extreme_z > 5).sum()}')
print('Nota: En expresión génica, valores extremos pueden ser biológicamente relevantes.')

---
## 7. Análisis Bivariante — Variables Clínicas vs. Diagnóstico
> **Pregunta EDA:** ¿Hay correlación entre las variables y la variable objetivo? ¿Cómo se distribuyen los datos en función de categorías?

In [ ]:
# --- 7.1 Variables numéricas por clase ---
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sano      = df[df[TARGET] == 0][col].dropna()
    psoriasis = df[df[TARGET] == 1][col].dropna()

    axes[i].hist(sano,      bins=25, alpha=0.6, color='steelblue', label='Sano',      edgecolor='white')
    axes[i].hist(psoriasis, bins=25, alpha=0.6, color='tomato',    label='Psoriasis', edgecolor='white')

    # Test estadístico de diferencia
    stat, pval = stats.mannwhitneyu(sano, psoriasis, alternative='two-sided')
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
    axes[i].set_title(f'{col} [{sig} p={pval:.3f}]')
    axes[i].legend()

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución por Diagnóstico — Variables Clínicas\n(Mann-Whitney U test)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_bivar_numeric.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 7.2 Variables categóricas vs Diagnóstico ---
fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index') * 100
    ct.columns = ['Sano', 'Psoriasis']
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'],
            edgecolor='white', rot=0)
    axes[i].set_title(f'{col} vs Diagnóstico')
    axes[i].set_ylabel('% dentro de categoría')
    axes[i].legend()

plt.suptitle('Variables Categóricas vs Diagnóstico', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_bivar_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 7.3 Boxplots comparativos (violín) ---
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

df_plot = df[num_cols + [TARGET]].copy()
df_plot[TARGET] = df_plot[TARGET].map({0: 'Sano', 1: 'Psoriasis'})

for i, col in enumerate(num_cols):
    sns.violinplot(data=df_plot, x=TARGET, y=col, ax=axes[i],
                   palette={'Sano': 'steelblue', 'Psoriasis': 'tomato'},
                   inner='box', cut=0)
    axes[i].set_title(col)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución por Clase — Violin Plots', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_violin.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Matriz de Correlación — Variables Clínicas

In [ ]:
# Codificar variable objetivo para incluirla en la correlación
df_corr = df[num_cols + [TARGET]].copy()
corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 10})
ax.set_title('Matriz de Correlación — Variables Clínicas\n(incluye variable objetivo)', fontsize=13)
plt.tight_layout()
plt.savefig('fig_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Variables más correlacionadas con el objetivo
corr_target = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print('=== CORRELACIÓN CON LA VARIABLE OBJETIVO ===')
print(corr_target.to_string())

---
## 9. Análisis de Expresión Génica vs. Diagnóstico

In [ ]:
# --- 9.1 Top genes más correlacionados con diagnóstico ---
from scipy.stats import pointbiserialr

gene_corr = {}
for gene in gene_cols:
    col_data = df[gene].fillna(df[gene].median())
    r, p = pointbiserialr(df[TARGET], col_data)
    gene_corr[gene] = {'r': r, 'p': p, 'r_abs': abs(r)}

gene_corr_df = pd.DataFrame(gene_corr).T.sort_values('r_abs', ascending=False)

print(f'Top 10 genes más correlacionados con diagnóstico:')
display(gene_corr_df.head(10).round(4))

In [ ]:
# --- 9.2 Volcano-style plot ---
gene_corr_df['-log10p'] = -np.log10(gene_corr_df['p'].clip(1e-300))

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(gene_corr_df['r'], gene_corr_df['-log10p'],
                     c=gene_corr_df['r'], cmap='coolwarm',
                     alpha=0.6, s=30, edgecolors='none')

ax.axhline(-np.log10(0.05), color='orange', linestyle='--', linewidth=1.2, label='p=0.05')
ax.axhline(-np.log10(0.001), color='red',   linestyle='--', linewidth=1.2, label='p=0.001')
ax.axvline(0, color='gray', linewidth=0.8)

# Etiquetar top genes
top5 = gene_corr_df.head(5)
for gene_name, row in top5.iterrows():
    ax.annotate(gene_name, (row['r'], row['-log10p']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

plt.colorbar(scatter, ax=ax, label='Correlación (r)')
ax.set_xlabel('Correlación punto-biserial (r)')
ax.set_ylabel('-log10(p-value)')
ax.set_title('Correlación Génica con Diagnóstico de Psoriasis')
ax.legend()
plt.tight_layout()
plt.savefig('fig_volcano_genes.png', dpi=150, bbox_inches='tight')
plt.show()

sig_genes = gene_corr_df[gene_corr_df['p'] < 0.05]
print(f'Genes con correlación significativa (p<0.05): {len(sig_genes)}')
print(f'Genes con correlación altamente significativa (p<0.001): {(gene_corr_df["p"]<0.001).sum()}')

In [ ]:
# --- 9.3 Heatmap de los top 30 genes más relevantes ---
top30_genes = gene_corr_df.head(30).index.tolist()

df_heatmap = df[top30_genes + [TARGET]].copy()
df_heatmap = df_heatmap.sort_values(TARGET)
df_norm    = (df_heatmap[top30_genes] - df_heatmap[top30_genes].min()) / \
             (df_heatmap[top30_genes].max() - df_heatmap[top30_genes].min())

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(df_norm.T, ax=ax, cmap='viridis',
            xticklabels=False, yticklabels=True,
            cbar_kws={'label': 'Expresión normalizada'})
ax.set_title('Expresión de los Top 30 Genes más Correlacionados\n(ordenado por diagnóstico)',
             fontsize=13)
ax.set_xlabel('Pacientes (ordenados: Sano → Psoriasis)')

# Línea divisoria entre clases
n_sanos = (df_heatmap[TARGET] == 0).sum()
ax.axvline(n_sanos, color='red', linewidth=2, linestyle='--', label='Límite diagnóstico')

plt.tight_layout()
plt.savefig('fig_heatmap_top_genes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Análisis Multivariante — PCA de Expresión Génica

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Imputar faltantes y normalizar
X_genes = df[gene_cols].fillna(df[gene_cols].median())
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_genes)

# PCA
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
var_exp     = pca.explained_variance_ratio_ * 100
var_cum     = np.cumsum(var_exp)
n90         = np.argmax(var_cum >= 90) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 21), var_exp[:20], color='steelblue', edgecolor='white')
axes[0].plot(range(1, 21), var_cum[:20], 'r-o', markersize=4, label='Var. acumulada')
axes[0].axhline(90, color='orange', linestyle='--', label='90%')
axes[0].set_title('Scree Plot — PCA (Genes)')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].legend()

# PC1 vs PC2 coloreado por diagnóstico
labels = df[TARGET].values
colors_map = {0: 'steelblue', 1: 'tomato'}
for cls, color in colors_map.items():
    mask = labels == cls
    label_text = 'Sano' if cls == 0 else 'Psoriasis'
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=label_text, alpha=0.6, s=25, edgecolors='none')

axes[1].set_title(f'PCA — PC1 vs PC2\n(PC1={var_exp[0]:.1f}%, PC2={var_exp[1]:.1f}%)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Componentes necesarios para explicar 90% de la varianza: {n90}')
print(f'Reducción dimensional: {len(gene_cols)} genes → {n90} componentes')

---
## 11. Preprocesamiento Propuesto

In [ ]:
print('=' * 65)
print('RESUMEN DEL PLAN DE PREPROCESAMIENTO')
print('=' * 65)

plan = {
    'Valores faltantes — variables clínicas': [
        'Numéricas (<5% faltantes): imputar con mediana',
        'Categóricas: imputar con moda o crear categoría "Desconocido"',
        'Variables con >20% faltantes: crear indicador binario'
    ],
    'Valores faltantes — genes': [
        'Imputar con mediana por gen (preserva distribución)',
        'Eliminar genes con >30% de faltantes (si aplica)'
    ],
    'Valores atípicos': [
        'Variables clínicas: winsorizar al percentil 1-99',
        'Genes: conservar atípicos (pueden ser biológicamente relevantes)',
        'Evaluar con el equipo médico valores extremos en score PASI'
    ],
    'Transformaciones': [
        'Aplicar log(x+1) a variables con skewness > 1',
        'Aplicar log(x+1) a expresión génica (distribución lognormal)'
    ],
    'Codificación categórica': [
        'Variables binarias (Si/No): Label Encoding 0/1',
        'Sexo: Label Encoding',
        'Variables ordinales: Ordinal Encoding si aplica'
    ],
    'Normalización': [
        'StandardScaler para variables clínicas numéricas',
        'StandardScaler para genes (post-imputación)'
    ],
    'Balance de clases': [
        f'Ratio detectado: {(df[TARGET]==1).sum()} psoriasis vs {(df[TARGET]==0).sum()} sanos',
        'Si ratio > 2:1 → aplicar SMOTE o class_weight en la red neuronal'
    ]
}

for seccion, items in plan.items():
    print(f'\n▶ {seccion}')
    for item in items:
        print(f'   • {item}')

---
## 12. Conclusiones del EDA

> *Completa esta celda con tus conclusiones específicas una vez que hayas corrido el notebook con tu dataset real.*

### 12.1 Calidad de los Datos
- **Valores faltantes**: [Describe el patrón encontrado, si es aleatorio (MCAR) o sistemático (MAR/MNAR)]
- **Valores atípicos**: [Variables afectadas y si son errores de captura o valores biológicamente posibles]

### 12.2 Distribuciones y Transformaciones
- [Indica qué variables requieren transformación logarítmica y por qué]
- [Menciona si la expresión génica sigue distribución lognormal, lo cual es típico en datos de microarreglos o RNA-seq]

### 12.3 Balance de Clases
- [Especifica si hay desbalance y la estrategia seleccionada]

### 12.4 Relevancia de Variables
- [Variables clínicas con mayor diferencia entre clases (prueba estadística significativa)]
- [Top genes más correlacionados con el diagnóstico]
- [Si el PCA separa visualmente las clases — indica separabilidad lineal]

### 12.5 Cardinalidad
- [Variables categóricas sin alta cardinalidad → no se requiere técnica especial]
- [Si alguna variable tuviera muchas categorías → hashing o embeddings]

### 12.6 Hallazgos Clave
1. **Tendencia principal**: [Ej. "Los genes GENE_XXXX y GENE_YYYY muestran la mayor diferencia de expresión entre grupos"]
2. **Variable clínica más discriminante**: [Ej. "El score PASI y la PCR muestran diferencias estadísticamente significativas entre sanos y pacientes"]
3. **Desafío para el modelado**: [Ej. "La alta dimensionalidad genómica (200+ genes) frente a N=500 pacientes requiere regularización fuerte o reducción dimensional"]
4. **Recomendación de arquitectura**: [Ej. "Considerar red neuronal con dropout y batch normalization dado el perfil del dataset"]